# Semantic Search Engine — Project Overview

## What this project builds

A search engine that understands *meaning*, not just keywords. You type a query in plain English and it finds the most relevant articles — even when they use completely different words than your query.

We also build a classic keyword search system (BM25) alongside it, so you can compare both side by side and see exactly where each method wins and fails.

---

## Why this matters

Embeddings are the single most important concept in modern AI engineering. Every RAG system, every AI agent with memory, every document chatbot, every recommendation engine — all of them depend on exactly what you will build here. This project gives you direct, hands-on intuition for how it works.

---

## Dataset — AG News (1,000 articles)

| Column | Description |
|---|---|
| `content` | 1–3 sentences of real news text |
| `category` | One of: Business, Technology, Sports, World |

We use 1,000 articles — small enough to run fast on a laptop, large enough to produce meaningful search results.

---

## Project structure

| File | Purpose |
|---|---|
| `01_data_exploration.ipynb` | Understand and clean the dataset |
| `02_embeddings.ipynb` | Convert articles to vectors and save to disk |
| `03_search_engines.ipynb` | Build FAISS semantic search + BM25 keyword search |
| `04_comparison.ipynb` | Compare both systems side by side |
| `app.py` | Streamlit web interface |

---

## Notebook 1 — Data Exploration

**Goal:** Load the AG News dataset, understand its structure, check data quality, and prepare a clean version for embedding.

**What you will do:**
- Load the CSV and inspect rows and column types
- Count articles per category
- Measure article lengths (important before embedding)
- Check for missing values and duplicates
- Read sample articles from each category
- Save a clean dataset for the next notebook

**Input:** `ag_news_1000.csv`  
**Output:** cleaned `ag_news_1000.csv` (2 columns, no duplicates, no nulls)

# Imports and Setup

In [1]:
# Imports and setup

import os
import warnings
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"   # suppress TensorFlow C++ logs
warnings.filterwarnings("ignore")           # suppress Python-level warnings

# pandas: the standard library for working with tables (like Excel, but in Python)
import pandas as pd

# numpy: the standard library for working with numbers and arrays
import numpy as np

print("✅ Libraries loaded.")
print(f"   pandas  version: {pd.__version__}")
print(f"   numpy   version: {np.__version__}")

✅ Libraries loaded.
   pandas  version: 2.3.3
   numpy   version: 1.26.4


# Load The Dataset

In [2]:
# Load the dataset

# pd.read_csv() reads a CSV file and stores it as a "DataFrame"


df = pd.read_csv("OneDrive/Documents/Projects/Semantic Search Engine/ag_news_1000.csv")

# .shape tells you (number of rows, number of columns)
print(f"Dataset shape: {df.shape}")
print(f"  → {df.shape[0]} articles, {df.shape[1]} columns\n")

# .columns lists the column names
print(f"Column names: {list(df.columns)}\n")

# .dtypes tells you the data type of each column
# 'object' means text (string), 'int64' means whole number
print("Column data types:")
print(df.dtypes)

Dataset shape: (1000, 2)
  → 1000 articles, 2 columns

Column names: ['content', 'category']

Column data types:
content     object
category    object
dtype: object


# Preview The Data

In [3]:
# Preview rows

print("First 5 rows of the dataset:")
print("=" * 60)

# We loop through the first 5 rows and print each one nicely
for i, row in df.head(5).iterrows():
    # i = row number (0, 1, 2, ...)
    # row["category"] = the category label
    # row["content"]  = the article text
    # [:200] = only show first 200 characters (articles are long)
    print(f"\nRow {i} | Category: [{row['category']}]")
    print(f"Text: {row['content'][:200]}...")
    print("-" * 40)

First 5 rows of the dataset:

Row 0 | Category: [Business]
Text: Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again....
----------------------------------------

Row 1 | Category: [Business]
Text: Carlyle Looks Toward Commercial Aerospace (Reuters) Reuters - Private investment firm Carlyle Group,\which has a reputation for making well-timed and occasionally\controversial plays in the defense in...
----------------------------------------

Row 2 | Category: [Business]
Text: Oil and Economy Cloud Stocks' Outlook (Reuters) Reuters - Soaring crude prices plus worries\about the economy and the outlook for earnings are expected to\hang over the stock market next week during t...
----------------------------------------

Row 3 | Category: [Business]
Text: Iraq Halts Oil Exports from Main Southern Pipeline (Reuters) Reuters - Authorities have halted oil export\flows from the main pipeline in southern I

# Understand the categories

In [4]:
# Category distribution


# .value_counts() counts how many times each unique value appears
# This tells us how many articles belong to each category
category_counts = df["category"].value_counts()

print("Number of articles per category:")
print("=" * 35)

# Loop through each category and print a simple bar
for category, count in category_counts.items():
    # Build a simple ASCII bar chart using "█" characters
    bar = "█" * (count // 10)    # one block per 10 articles
    print(f"  {category:<12} {count:>4}  {bar}")

print()
print(f"Total articles: {len(df)}")
print()

# .nunique() tells you how many UNIQUE values exist
print(f"Number of unique categories: {df['category'].nunique()}")

# .unique() lists all unique values
print(f"Category names: {list(df['category'].unique())}")

Number of articles per category:
  Technology    472  ███████████████████████████████████████████████
  World         212  █████████████████████
  Business      174  █████████████████
  Sports        142  ██████████████

Total articles: 1000

Number of unique categories: 4
Category names: ['Business', 'Technology', 'Sports', 'World']


# Understand the text content

In [5]:
# Text length analysis

# Before embedding text, we need to know how long each article is.
# Sentence transformer models work best on shorter texts (< 512 words).
# Let's check our data is within a reasonable range.

# .apply() runs a function on every row of a column
# len() counts the number of characters in a string
df["text_length"] = df["content"].apply(len)

# len() on a string counts characters
# .split() breaks a string into words by spaces — len() then counts words
df["word_count"] = df["content"].apply(lambda x: len(x.split()))

print("Text length statistics (characters):")
print("-" * 40)
# .describe() gives you count, mean, min, max, and percentiles
print(df["text_length"].describe().round(1))

print("\nWord count statistics:")
print("-" * 40)
print(df["word_count"].describe().round(1))

# Find the shortest and longest articles
print(f"\nShortest article ({df['word_count'].min()} words):")
shortest = df.loc[df["word_count"].idxmin(), "content"]
print(f"  {shortest[:150]}")

print(f"\nLongest article ({df['word_count'].max()} words):")
longest = df.loc[df["word_count"].idxmax(), "content"]
print(f"  {longest[:150]}...")

# Check how many articles are within safe embedding limits
# The model we use (all-MiniLM-L6-v2) handles up to ~256 words well
safe = df[df["word_count"] <= 256]
print(f"\nArticles under 256 words: {len(safe)} / {len(df)} "
      f"({len(safe)/len(df)*100:.1f}%)")

Text length statistics (characters):
----------------------------------------
count    1000.0
mean      250.2
std       111.2
min       100.0
25%       166.0
50%       242.0
75%       290.2
max       862.0
Name: text_length, dtype: float64

Word count statistics:
----------------------------------------
count    1000.0
mean       39.9
std        17.5
min        11.0
25%        27.0
50%        38.0
75%        47.0
max       134.0
Name: word_count, dtype: float64

Shortest article (11 words):
  Buffy the Censor Slayer &lt;strong&gt;Letters&lt;/strong&gt; Readers drive stake through parents' group

Longest article (134 words):
  Technology as Fashion Analyzing the success of the iPod mini in Japan, JapanConsuming writes,  #147;The iPod mini is in fact one of those all too rare...

Articles under 256 words: 1000 / 1000 (100.0%)


# Check for missing or duplicate data

In [6]:
# Data quality check

print("Checking for missing values (NaN):")
print("-" * 35)
# .isnull().sum() counts the number of empty/missing cells per column
print(df.isnull().sum())

print("\nChecking for duplicate articles:")
print("-" * 35)
# .duplicated() marks rows that are exact copies of a previous row
# .sum() counts how many such duplicates exist
num_duplicates = df.duplicated(subset=["content"]).sum()
print(f"  Duplicate articles found: {num_duplicates}")

if num_duplicates == 0:
    print("  ✅ No duplicates — data is clean!")
else:
    print(f"  ⚠️  Removing {num_duplicates} duplicates...")
    df = df.drop_duplicates(subset=["content"]).reset_index(drop=True)
    print(f"  ✅ Clean dataset now has {len(df)} rows.")

# Reset index adds a clean 0,1,2,3... index after any removals
df = df.reset_index(drop=True)

print(f"\nFinal clean dataset: {df.shape[0]} rows, {df.shape[1]} columns")
print("Ready to proceed to embedding!")

Checking for missing values (NaN):
-----------------------------------
content        0
category       0
text_length    0
word_count     0
dtype: int64

Checking for duplicate articles:
-----------------------------------
  Duplicate articles found: 0
  ✅ No duplicates — data is clean!

Final clean dataset: 1000 rows, 4 columns
Ready to proceed to embedding!


# Look at example articles per category

In [8]:
# Sample articles from each category
# This is important: we need to UNDERSTAND our data before building
# a search engine. Let's read one example from each category.

print("One sample article from each category:")
print("=" * 60)

for category in df["category"].unique():
    # Filter to only rows where category matches
    # .iloc[0] takes the first result
    sample = df[df["category"] == category]["content"].iloc[0]
    
    print(f"\n📂 Category: {category}")
    print(f"   {sample[:250]}...")
    print("-" * 50)

One sample article from each category:

📂 Category: Business
   Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again....
--------------------------------------------------

📂 Category: Technology
   'Madden,' 'ESPN' Football Score in Different Ways (Reuters) Reuters - Was absenteeism a little high\on Tuesday among the guys at the office? EA Sports would like\to think it was because "Madden NFL 2005" came out that day,\and some fans of the footba...
--------------------------------------------------

📂 Category: Sports
   Phelps, Thorpe Advance in 200 Freestyle (AP) AP - Michael Phelps took care of qualifying for the Olympic 200-meter freestyle semifinals Sunday, and then found out he had been added to the American team for the evening's 400 freestyle relay final. Phe...
--------------------------------------------------

📂 Category: World
   Venezuelans Vote Early in Referendum on Chavez Rule (Re

In [9]:
# Drop helper columns before saving
# We added text_length and word_count just for analysis.
# Drop them so the saved CSV stays clean with only 2 columns.

df = df.drop(columns=["text_length", "word_count"])
print(f"Columns after cleanup: {list(df.columns)}")

Columns after cleanup: ['content', 'category']


# Save The Clean Dataset and Summary

In [11]:
# Save clean data and summary

# Save the cleaned dataframe back to CSV so all future notebooks
# use the same clean version.

df.to_csv("OneDrive/Documents/Projects/Semantic Search Engine/ag_news_1000.csv", index=False)

# Reload it immediately to confirm what was actually saved
df_saved = pd.read_csv("OneDrive/Documents/Projects/Semantic Search Engine/ag_news_1000.csv")

print("✅ Clean dataset saved to ag_news_1000.csv")
print()
print("=" * 50)
print("NOTEBOOK 1 COMPLETE — Summary")
print("=" * 50)
print(f"  Total articles     : {len(df_saved)}")
print(f"  Columns            : {list(df_saved.columns)}")
print(f"  Categories         : {list(df_saved['category'].unique())}")
print(f"  Missing values     : {df_saved.isnull().sum().sum()}")
print(f"  Duplicates         : {df_saved.duplicated().sum()}")
print()
print("Next step: Open 02_embeddings.ipynb")

✅ Clean dataset saved to ag_news_1000.csv

NOTEBOOK 1 COMPLETE — Summary
  Total articles     : 1000
  Columns            : ['content', 'category']
  Categories         : ['Business', 'Technology', 'Sports', 'World']
  Missing values     : 0
  Duplicates         : 0

Next step: Open 02_embeddings.ipynb


---

## Notebook 1 — Summary & Key Observations

### What we found

| Property | Value |
|---|---|
| Total articles | 1,000 |
| Columns | `content`, `category` |
| Missing values | None |
| Duplicates | None |
| Average article length | ~40 words |
| Longest article | 134 words |
| All articles under 256 words | Yes (100%) |

The dataset is clean and ready for embedding. No preprocessing was needed beyond removing the temporary helper columns.

---

### Category distribution — an important observation

| Category | Count | Notes |
|---|---|---|
| Technology | 472 | Largest group — nearly half the dataset |
| World | 212 | |
| Business | 174 | |
| Sports | 142 | Smallest group |

The dataset is **imbalanced** — Technology articles dominate. This means search results will naturally skew toward Technology articles unless a query is very specific to another category. This is worth keeping in mind when we evaluate results in Notebook 4.

---

### Real-world data messiness — the Madden example

One Technology-labelled article was about *Madden NFL 2005* — a football video game. It was categorised as Technology (a video game product), but its content is largely about football and sports culture.

This is normal and expected in real datasets. Category labels reflect the *editorial decision* of the news agency, not always the dominant topic of the text. A semantic search engine will embed this article closer to Sports content than to, say, a semiconductor article — because the *words and meaning* are more sports-adjacent.

**What this means for our project:** When we evaluate results in Notebook 4, we should not treat category mismatches as automatic failures. A result that crosses category boundaries may still be semantically valid. This is actually one of the advantages of semantic search — it finds conceptual relevance that rigid category labels miss.

---

### What article length tells us

Average article length is ~40 words. Our embedding model (`all-MiniLM-L6-v2`) handles up to ~256 words comfortably — all 1,000 articles fit well within that limit. No truncation will occur.

---

### Next step

Open **`02_embeddings.ipynb`** to convert all 1,000 articles into 384-dimensional vectors using a sentence transformer model.